# PhoBERT NER Baseline
Fine-tune `vinai/phobert-base-v2` voi `AutoModelForTokenClassification` (num_labels=3).

**Nhan:** `"O": 0, `"B-COMP"`: 1, `"I-COMP"`: 2

In [ ]:
import json
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    set_seed,
)
from seqeval.metrics import precision_score, recall_score, f1_score

set_seed(42)

In [ ]:
LABEL_LIST = ["O", "B-COMP", "I-COMP"]

LABEL2ID = {
    "O": 0,
    "B-COMP": 1,
    "I-COMP": 2,
}

ID2LABEL = {
    0: "O",
    1: "B-COMP",
    2: "I-COMP",
}

MODEL_NAME = "vinai/phobert-base-v2"
MAX_LEN = 256

## 1. Tai du lieu

In [ ]:
def load_ner_json(path):
    """Doc file JSON chua danh sach mau NER."""
    with open(path, encoding="utf-8") as f:
        records = json.load(f)

    tokens = []
    ner_tags = []
    for r in records:
        tokens.append(r["tokens"])
        labels = r.get("labels", r.get("ner_tags", []))
        ner_tags.append([LABEL2ID[lbl] for lbl in labels])

    return Dataset.from_dict({
        "tokens": tokens,
        "ner_tags": ner_tags,
    })


train_dataset = load_ner_json("../data/processed/ner_train.json")
test_dataset = load_ner_json("../data/processed/ner_test.json")

print(f"Train: {len(train_dataset)} mau")
print(f"Test : {len(test_dataset)} mau")
print(f"Mau vi du:")
print(train_dataset[0])

## 2. Load PhoBERT Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

## 3. Ham tokenize_and_align_labels — YEU CAU BAT BUOC

**Rang buoc ky thuat:**

1. `is_split_into_words=True` — bao cho tokenizer biet dau vao da duoc tach tu
2. Duyet `word_ids()` de xac dinh token nao la dau cua tu, token nao la phu (sub-word)
3. Special token (`None`) va sub-word -> gan `-100`
4. Chi token dau tien cua moi tu -> giu nhan that

In [ ]:
def tokenize_and_align_labels(examples):
    """
    Tokenize va gan nhan cho NER, dung cach dung voi `is_split_into_words=True`.

    Moi token (sub-word) duoc gan:
      - -100  : special token (cls/sep/pad) hoac sub-word phu
      - nhan thuc te (0/1/2) : chi token dau tien cua moi tu
    """
    # is_split_into_words=True : tokenizer biet dau vao da la danh sach tu
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_offsets_mapping=False,
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(i)  # list gia tri word_id moi sub-word
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                # Special token (cls, sep, pad) -> bo qua
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Token dau tien cua tu -> gan nhan thuc te
                label_ids.append(labels[word_idx])
                previous_word_idx = word_idx
            else:
                # Sub-word cua tu da xu ly roi -> bo qua
                label_ids.append(-100)

        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names,
)

test_tokenized = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names,
)

print(f"Train tokenized: {len(train_tokenized)} mau")
print(f"Test tokenized : {len(test_tokenized)} mau")
print(f"\nMau[0] input_ids[:15] : {train_tokenized[0]['input_ids'][:15]}")
print(f"Mau[0] labels[:15]   : {train_tokenized[0]['labels'][:15]}")

## 4. Load mo hinh

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

## 5. Ham compute_metrics — YEU CAU BAT BUOC

**Rang buoc ky thuat:**

1. Vong lap `for` loc bo hoan toan cac gia tri `-100` khoi predictions va labels
2. Truyen danh sach da loc vao `seqeval`
3. Dung `zero_division=0` de tranh warning khi khong co mau hop le

In [ ]:
def compute_metrics(eval_pred):
    """
    Tinh seqeval metrics (precision, recall, F1) cho NER.

    **Buoc loc -100 bat buoc:**
    Chi cac token co nhan thuc te (khong phai -100) moi duoc dua vao
    ham tinh diem cua seqeval.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    # --- Vong lap loc bo -100 (BAT BUOC) ---
    for prediction, label in zip(predictions, labels):
        current_predictions = []
        current_labels = []

        for pred, lab in zip(prediction, label):
            if lab != -100:  # Chi giu lai nhan hop le
                current_predictions.append(ID2LABEL[pred])
                current_labels.append(ID2LABEL[lab])

        true_predictions.append(current_predictions)
        true_labels.append(current_labels)

    precision = precision_score(true_labels, true_predictions, zero_division=0)
    recall = recall_score(true_labels, true_predictions, zero_division=0)
    f1 = f1_score(true_labels, true_predictions, zero_division=0)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## 6. DataCollator

Dung `DataCollatorForTokenClassification` voi `is_split_into_words=True`
de collator khong re-tokenize (khong pha vo alignment).

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    is_split_into_words=True,  # Quan trong: khong re-tokenize
)

## 7. Cau hinh TrainingArguments

**Luu y:** Khong luu checkpoint de tranh nen Git.

In [ ]:
training_args = TrainingArguments(
    output_dir="./ner_checkpoints",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="no",      # Khong luu checkpoint
    save_total_limit=0,     # Xoa tat ca checkpoint cu

    load_best_model_at_end=False,

    dataloader_pin_memory=False,
    logging_steps=50,
    report_to="none",
    seed=42,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 8. Huan luyen

In [ ]:
trainer.train()

## 9. Danh gia & In ket qua

In [ ]:
results = trainer.evaluate()

print("=" * 50)
print("PhoBERT NER Baseline")
print("=" * 50)
print(f"Precision : {results['eval_precision']:.4f}")
print(f"Recall    : {results['eval_recall']:.4f}")
print(f"F1-score  : {results['eval_f1']:.4f}")
print(f"Eval Loss : {results['eval_loss']:.4f}")